## Bronze Layer — Raw Ingestion

This notebook is the entry point for the Movie Analytics pipeline. It takes raw CSV files from the ADLS landing zone and loads them into Delta Lake with minimal transformation — just enough to make the data queryable and traceable.

**What comes in:** 5 CSV files from `/movies_data/` — `cast`, `crew`, `genres`, `movies`, and `reviews`.

**What goes out:** 5 Delta tables in `employeedatacatalog.bronze_movie`, each stamped with ingestion metadata.

**Steps:**
1. **Set up paths and schemas** — ADLS storage paths for each medallion layer + Unity Catalog schema creation
2. **Read CSVs** — Load all `.csv` files with `inferSchema` and quote-escape handling (the reviews CSV has commas embedded in free-text content, which causes some rows to parse incorrectly)
3. **Add bronze metadata** — Four audit columns on every row: ingestion timestamp, source file path, batch ID, and a basic validity flag
4. **Write to Delta** — Persist to ADLS as Delta tables and register in Unity Catalog with `overwriteSchema` to handle type drift between runs

**Known issue:** The reviews CSV has commas inside the `content` field that the CSV parser can't fully handle, even with escape quoting. Some review rows end up with content fragments scattered across the wrong columns. These malformed rows are flagged but not dropped here — that cleanup happens in the silver layer.

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# All our movie data lives in ADLS under the 'employee' container
root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# Unity Catalog schema names — one per medallion layer
bronze_sch = "bronze_movie"
silver_sch = "silver_movie"
gold_sch = "gold_movie"

movies_db = "employeedatacatalog"

# Make sure all three schemas exist before we start writing
schema_data = [bronze_sch, silver_sch, gold_sch]

for sch in schema_data:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {movies_db}.{sch}")

In [0]:
# Source folder with all the raw movie CSVs (cast, crew, genres, movies, reviews)
source_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/movies_data/"

csv_files = [file for file in dbutils.fs.ls(source_path) if file.name.endswith('.csv')]

dataframes = {}

# Read each CSV with inferSchema and quote-escaping enabled.
# The reviews CSV has commas inside content fields, so the escape option
# helps — though some rows are still malformed (handled in silver).
for file in csv_files:
    name = file.name.replace('.csv', '').replace(' ', '_')
    dataframes[name] = spark.read.option("header", True).option("inferSchema", True).option("escape", '"').csv(file.path)

print(f"Loaded {len(dataframes)} tables: {list(dataframes.keys())}")

In [0]:
# Stamp each dataframe with ingestion metadata so we can trace lineage later:
#   _bronze_ingested_at  — when this row was loaded
#   _bronze_source_file  — which CSV file it came from
#   _bronze_batch_id     — unique ID for this load run
#   _bronze_is_valid     — basic null check on the first column (quick sanity flag)

bronze_dfs = {}

for name, df in dataframes.items():
    first_col = df.columns[0]
    
    bronze_dfs[name] = df.withColumn("_bronze_ingested_at", current_timestamp())\
                         .withColumn("_bronze_source_file", col("_metadata.file_path"))\
                         .withColumn("_bronze_batch_id", expr("uuid()"))\
                         .withColumn("_bronze_is_valid", col(first_col).isNotNull())

for name, df in bronze_dfs.items():
    print(f"{name}: {len(df.columns)} columns (validity check on '{df.columns[0]}')")

In [0]:
# Persist each bronze dataframe as a Delta table and register it in Unity Catalog.
# Using overwriteSchema in case column types changed between runs (e.g. cast_order
# flipped between string and int depending on the CSV inferSchema result).

for name, df in bronze_dfs.items():
    table_path = f"{bronze_path}/{name.lower()}"
    
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(table_path)

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {movies_db}.{bronze_sch}.{name}
                  USING DELTA LOCATION '{table_path}'""")
    print(f"Saved: {movies_db}.{bronze_sch}.{name}")